# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all record sets and their fields with '@id'

record_sets = meta.record_sets
if not record_sets:
    print("No record sets found in the metadata.")
else:
    print("Record Sets:")
    for rset in record_sets:
        print(f"- RecordSet name: {rset.name}, @id: {rset.id}")
        print("  Fields and their @id:")
        # Fields in RecordSet might be found in rset.fields
        fields = getattr(rset, 'fields', [])
        for field in fields:
            print(f"    - {getattr(field, 'name', '')} (@id: {getattr(field, 'id', '')})")

## 3. Data Extraction
Load data from record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from all record sets

record_sets = meta.record_sets
dataframes = {}
if not record_sets:
    print("No record sets to extract records from.")
else:
    for rset in record_sets:
        rset_id = rset.id
        try:
            records = list(dataset.records(record_set=rset_id))
            if records:
                df = pd.DataFrame(records)
                dataframes[rset_id] = df
                print(f"Loaded DataFrame for record set {rset.name} (@id: {rset_id})")
                print(df.head())
                print('---')
        except Exception as e:
            print(f"Could not load record set {rset_id} due to error: {e}")

# Demonstrate with the first available record set (if present):
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    print(f"Columns in the first record set DataFrame ({first_rs_id}):")
    print(dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())
else:
    print('No data available in any record set.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section may include operations like removing outliers, transforming distributions, or grouping by key attributes.

In [ ]:
# Choose a numeric field and group field from the first DataFrame if available

if dataframes:
    df = dataframes[first_rs_id]
    # Try to detect some numeric fields (float or int fields)
    numeric_candidates = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
    group_candidates = df.select_dtypes(include=['object']).columns.tolist()
    print(f"Numeric field candidates: {numeric_candidates}")
    print(f"Group field candidates: {group_candidates}")
    
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by another field if exists
        if group_candidates:
            group_field = group_candidates[0]
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            print(grouped_df.head())
    else:
        print("No numeric fields detected for analysis.")
else:
    print("No DataFrame available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_candidates:
    # Histogram for the numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    # Boxplot by group if possible
    if group_candidates:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Not enough data for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

**Summary:**
- The FAIR<sup>2</sup> mass spectrometry dataset schema was loaded successfully with `mlcroissant`.
- Record set, fields, and their `@id`s can be discovered programmatically.
- Basic EDA was performed on available numeric fields, including filtering, normalization, and grouping.
- Distributions and groupwise statistics can be visualized for further insights.

For deeper domain-specific analysis, consult the Croissant schema definitions, field descriptions, and linked documentation.